# 14.8 - Stateful Agent

Status: VERIFIED

## What Are We Solving?
A stateful agent maintains persistent state across multiple user turns, tracking goals, progress, and preferences beyond simple conversation history.

In [1]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # loads from .env in project root
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "qwen/qwen3.8-27b"

# Quick test
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'groq ok' only"}],
    max_tokens=10
)
print(f"Groq connected: {r.choices[0].message.content.strip()}")
print(f"Model: {MODEL}")

Groq connected: groq ok
Model: qwen/qwen3.8-27b


## State Management

In [2]:
from dataclasses import dataclass, field, asdict

@dataclass
class AgentState:
    goal: str = ""
    completed: list = field(default_factory=list)
    results: dict = field(default_factory=dict)
    status: str = "idle"

class StatefulAgent:
    def __init__(self):
        self.state = AgentState()
    
    def run(self, user_input: str) -> str:
        self.state.goal = user_input
        self.state.status = "active"
        
        # Build context from state
        context = f"Goal: {self.state.goal}\nCompleted: {self.state.completed}"
        
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": f"You are a stateful assistant.\n{context}\nDecide the next step or provide a final answer."},
                {"role": "user", "content": user_input}
            ]
        )
        answer = response.choices[0].message.content
        
        self.state.completed.append({"input": user_input[:50], "response": answer[:100]})
        self.state.status = "done"
        
        return answer

agent = StatefulAgent()
r1 = agent.run("Research Python async patterns")
print(f"Turn 1: {r1[:100]}...")
r2 = agent.run("Now summarize what we found")
print(f"Turn 2: {r2[:100]}...")
print(f"State: {len(agent.state.completed)} interactions logged")

Turn 1: ```python
import asyncio
import threading
import time
from typing import AsyncGenerator
from collect...


Turn 2: ```python
import asyncio
import threading
import time
from typing import AsyncGenerator

class Async...
State: 2 interactions logged


In [3]:
# Verification
assert len(agent.state.completed) == 2
assert agent.state.status == "done"
print("VERIFICATION PASSED: Phase 14.8 complete")

VERIFICATION PASSED: Phase 14.8 complete
